# XJTU-SY Bearing Dataset Analysis
## Run-to-Failure Validation Across 3 Operating Conditions

The XJTU-SY dataset from Xi'an Jiaotong University contains 15 run-to-failure experiments on LDK UER204 bearings across 3 operating conditions (varying speed and load). This is the most diverse dataset in the benchmark — it covers multiple failure modes (inner race, outer race, cage, combined) and exhibits extreme lifetime variation, from Bearing2_4 at just 42 minutes to Bearing3_1 at 2,538 minutes (~42 hours). The 15 bearings provide the strongest test of whether the adaptive PID framework generalizes across operating regimes and failure types.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
from pathlib import Path
import sys
sys.path.insert(0, '..')

from datasets.xjtu_sy.config import XJTU_SY_CONFIG

plt.style.use('seaborn-v0_8-whitegrid')
CB_PALETTE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00"]

# Load metrics if available
metrics_path = Path("../analysis/xjtu_sy_metrics.csv")
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    print(f"Loaded {len(metrics)} result rows")
else:
    metrics = None
    print("XJTU-SY metrics not yet available. Run the benchmark first.")

## Dataset Overview

In [ ]:
rows = []
for name, info in XJTU_SY_CONFIG["bearing_failures"].items():
    cond_num = int(name.split("_")[0].replace("Bearing", ""))
    cond = XJTU_SY_CONFIG["conditions"][cond_num]
    rows.append({
        "Bearing": name,
        "Condition": cond_num,
        "RPM": cond["rpm"],
        "Load (kN)": cond["radial_load_kn"],
        "Failure Mode": info["failure"],
        "Life (min)": info["life_min"],
        "Life (hours)": round(info["life_min"] / 60, 1),
    })
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## OEM Specification Extraction

In [ ]:
params_path = Path("../analysis/extracted_oem_params.json")
if params_path.exists():
    with open(params_path) as f:
        all_params = json.load(f)
    uer204 = all_params.get("UER204", {})
    print("LDK UER204 — Extracted OEM Parameters")
    print(f"  Bore: {uer204.get('bore_mm', 'N/A')} mm")
    print(f"  Dynamic load rating (C): {uer204.get('C_kn', 'N/A')} kN")
    print(f"  Confidence: {uer204.get('extraction_confidence', 'N/A')}")
    print(f"  Source: {uer204.get('source_file', 'N/A')}")
    
    # Ground truth comparison
    specs = XJTU_SY_CONFIG["oem_specs"]
    print(f"\nGround truth: C = {specs['C_kn']} kN, bore = {specs['bore_mm']} mm")
else:
    print("Extraction results not available. Run RAG pipeline first.")

## L10 Life Calculations

In [ ]:
from core.oem_prior import compute_l10_hours

specs = XJTU_SY_CONFIG["oem_specs"]
fig, ax = plt.subplots(figsize=(10, 6))

for cond_num, cond in XJTU_SY_CONFIG["conditions"].items():
    l10h = compute_l10_hours(C_kn=specs["C_kn"], P_kn=cond["radial_load_kn"],
                              rpm=cond["rpm"], p=specs["life_exponent"])
    
    # Get actual lifetimes for this condition
    lifetimes = []
    names = []
    for bname, binfo in XJTU_SY_CONFIG["bearing_failures"].items():
        if bname.startswith(f"Bearing{cond_num}_"):
            lifetimes.append(binfo["life_min"] / 60)
            names.append(bname)
    
    color = CB_PALETTE[cond_num - 1]
    ax.scatter(lifetimes, [cond_num] * len(lifetimes), color=color, s=80, zorder=3,
              label=f"Cond {cond_num}: {cond['rpm']} RPM, {cond['radial_load_kn']} kN")
    ax.axvline(l10h, color=color, linestyle='--', alpha=0.7)
    ax.text(l10h, cond_num + 0.15, f"L10={l10h:.1f}h", color=color, fontsize=9, ha='center')

ax.set_xlabel("Actual bearing lifetime (hours)")
ax.set_ylabel("Operating condition")
ax.set_yticks([1, 2, 3])
ax.legend(loc="upper right")
ax.set_title("Actual bearing lifetimes vs ISO 281 L10 prediction")
plt.tight_layout()
plt.savefig("../reports/figures/xjtu_sy_lifetime_vs_l10.png", dpi=150, bbox_inches="tight")
plt.show()

## Feature Trajectories

In [ ]:
# Check if cached features exist
processed_dir = Path("../data/processed")
selected = ["Bearing2_4", "Bearing1_1", "Bearing3_1"]  # 42min, 123min, 2538min

fig, axes = plt.subplots(1, len(selected), figsize=(15, 4), sharey=False)
for i, bearing_name in enumerate(selected):
    cache_path = processed_dir / f"xjtu_sy_{bearing_name}_features.csv"
    if cache_path.exists():
        df = pd.read_csv(cache_path)
        axes[i].plot(df["time_min"] / 60, df["kurtosis"], color=CB_PALETTE[i], linewidth=0.8)
        axes[i].set_xlabel("Time (hours)")
        axes[i].set_ylabel("Kurtosis")
        life_min = XJTU_SY_CONFIG["bearing_failures"][bearing_name]["life_min"]
        failure = XJTU_SY_CONFIG["bearing_failures"][bearing_name]["failure"]
        axes[i].set_title(f"{bearing_name}\n{life_min} min, {failure}")
    else:
        axes[i].text(0.5, 0.5, "Data not available", transform=axes[i].transAxes, ha='center')
        axes[i].set_title(bearing_name)

plt.suptitle("Kurtosis degradation trajectories — short, medium, and long life", y=1.02)
plt.tight_layout()
plt.savefig("../reports/figures/xjtu_sy_kurtosis_trajectories.png", dpi=150, bbox_inches="tight")
plt.show()

## Model Comparison

In [ ]:
if metrics is not None:
    # Aggregate by model
    model_summary = metrics.groupby("model").agg(
        mean_rmse=("rmse", "mean"),
        std_rmse=("rmse", "std"),
        mean_mae=("mae", "mean"),
    ).round(3).sort_values("mean_rmse")
    print("Model comparison (all 15 bearings):")
    print(model_summary.to_string())
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 5))
    models = model_summary.index.tolist()
    rmse_vals = model_summary["mean_rmse"].values
    colors = CB_PALETTE[:len(models)]
    ax.barh(models, rmse_vals, color=colors)
    ax.set_xlabel("Mean RMSE (hours)")
    ax.set_title("XJTU-SY: Model comparison")
    plt.tight_layout()
    plt.savefig("../reports/figures/xjtu_sy_model_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()
    
    # Per-condition breakdown
    print("\nPer-condition breakdown (PID + Regime):")
    pid_regime = metrics[metrics["model"] == "pid_regime"]
    if not pid_regime.empty:
        # Extract condition from unit_id
        pid_regime = pid_regime.copy()
        pid_regime["condition"] = pid_regime["unit_id"].str.extract(r"Bearing(\d)_").astype(int)
        cond_summary = pid_regime.groupby("condition").agg(
            mean_rmse=("rmse", "mean"),
            mean_mae=("mae", "mean"),
            n=("unit_id", "count"),
        ).round(3)
        print(cond_summary.to_string())
else:
    print("Run the benchmark to generate XJTU-SY metrics.")

## Cross-Manufacturer Comparison

In [ ]:
datasets = {
    "CWRU (SKF)": "../analysis/cwru_metrics.csv",
    "IMS (Rexnord)": "../analysis/ims_metrics.csv",
    "XJTU-SY (LDK)": "../analysis/xjtu_sy_metrics.csv",
}

comparison = []
for label, path in datasets.items():
    p = Path(path)
    if p.exists():
        df = pd.read_csv(p)
        pid_regime = df[df["model"] == "pid_regime"]
        if not pid_regime.empty:
            comparison.append({
                "Dataset": label,
                "Mean RMSE": round(pid_regime["rmse"].mean(), 3),
                "Mean MAE": round(pid_regime["mae"].mean(), 3),
                "N Trajectories": pid_regime["unit_id"].nunique(),
            })

if comparison:
    comp_df = pd.DataFrame(comparison)
    print("PID + Regime performance across manufacturers:")
    print(comp_df.to_string(index=False))
else:
    print("Not enough data for cross-manufacturer comparison.")

## Summary

Key findings from the XJTU-SY analysis:

1. **Extreme lifetime variation**: Bearing lifetimes range from 42 minutes to 2,538 minutes across conditions and failure modes, testing the framework's ability to adapt across very different degradation rates.
2. **Multi-condition validation**: Three operating conditions (varying RPM and radial load) exercise the L10 prior computation under different regimes — the OEM baseline must scale correctly with load and speed.
3. **Failure mode diversity**: Inner race, outer race, cage, and combined failures all appear in the dataset. The PID controller must track degradation regardless of the underlying physical mechanism.
4. **Cross-manufacturer generalization**: Comparing PID+Regime RMSE across SKF (CWRU), Rexnord (IMS), and LDK (XJTU-SY) bearings tests whether the adaptive framework transfers across manufacturers without retuning.